from re import compile
utok = np.array([b'\n !"%&\'()*+,-./0123456789:<=>@[]^_abcdefghijklmnopqrstuvwxyz{|}~']).view('B')
utok.tofile(f'py.{len(utok)}tok')
tok = np.full(2<<32-1,255,'B')
tok[tok] = range(len(utok))
tok[ord('\t')] = tok[ord(' ')]
tok[ord('A'):ord('Z')+1] = tok[ord('a'):ord('z')+1]
sub = {compile(r'\s*(#.*?)?\n'):',', compile('\n{2,}'):'\n', compile(r'\s+,'):',', '    ':' '}
a = open('py.txt', errors='ignore')
o = open(f'py.{len(utok)}', 'wb')
while _ := a.read(1<<24):
    for i,j in sub.items():
        _ = i.sub(j,_) if hasattr(i,'sub') else _.replace(i,j)
    _ = tok[np.array([_]).view('I')]
    _[_!=255].tofile(o)

import numpy as np
np.take(np.fromfile('py.64ent4.9','B'),_:=np.memmap('py.64',mode='r'),out=np.memmap('py.txt',mode='w+',shape=len(_)))

In [ ]:
import numpy as np
a = np.memmap('py.64', mode='r')

In [ ]:
C = []
I = []
t = a
for i in range(2, 6):
    t = np.bitwise_left_shift(t[:-1], 6, None if t.dtype=='B'else t[:-1], dtype='I')
    t |= a[i-1:]
    c = np.zeros(64**i, 'i')
    np.add.at(c, t, 1)
    i = c.nonzero()[0].astype('I')
    c = c[i]
    C.append(c)
    I.append(i)
del t

In [ ]:
O = []
for _ in range(65536-64):
    n,m = max(((i,c.argmax())for i,c in enumerate(C)), key=lambda ic:C[ic[0]][ic[1]]*(ic[0]+1))
    i = I[n][m]
    ii = i.item()
    c = C[n][m]
    if n == 0:
        O.append((ii>>6, ii&2**6-1))

        C[0][m] = 0

        t = I[1] & 2**12-1 != i
        t &= I[1] >> 6 != i
        I[1] = I[1][t]
        C[1] = C[1][t]

        t = I[2] & 2**12-1 != i
        t &= I[2] >> 6 & 2**12-1 != i
        t &= I[2] >> 12 != i
        I[2] = I[2][t]
        C[2] = C[2][t]

        t = I[3] & 2**12-1 != i
        t &= I[3] >> 6 & 2**12-1 != i
        t &= I[3] >> 12 & 2**12-1 != i
        t &= I[3] >> 18 != i
        I[3] = I[3][t]
        C[3] = C[3][t]
    elif n == 1:
        O.append((ii>>12, ii>>6&2**6-1, ii&2**6-1))
    
        t = I[0] == i & 2**12-1
        t |= I[0] == i >> 6
        C[0][t] -= c

        C[1][m] = 0

        t = I[2] & 2**18-1 != i
        t &= I[2] >> 6 != i
        I[2] = I[2][t]
        C[2] = C[2][t]

        t = I[3] & 2**18-1 != i
        t &= I[3] >> 6 & 2**18-1 != i
        t &= I[3] >> 12 != i
        I[3] = I[3][t]
        C[3] = C[3][t]
    elif n == 2:
        O.append((ii>>18, ii>>12&2**6-1, ii>>6&2**6-1, ii&2**6-1))
        
        t = I[0] == i & 2**12-1
        t |= I[0] == i >> 6 & 2**12-1
        t |= I[0] == i >> 12
        C[0][t] -= c

        t = I[1] == i & 2**18-1
        t |= I[1] == i >> 6
        C[1][t] -= c

        C[2][m] = 0

        t = I[3] & 2**24-1 != i
        t &= I[3] >> 6 != i
        I[3] = I[3][t]
        C[3] = C[3][t]
    elif n == 3:
        O.append((ii>>24, ii>>18&2**6-1, ii>>12&2**6-1, ii>>6&2**6-1, ii&2**6-1))
        
        t = I[0] == i & 2**12-1
        t |= I[0] == i >> 6 & 2**12-1
        t |= I[0] == i >> 12 & 2**12-1
        t |= I[0] == i >> 18
        C[0][t] -= c

        t = I[1] == i & 2**18-1
        t |= I[1] == i >> 6 & 2**18-1
        t |= I[1] == i >> 12
        C[1][t] -= c

        t = I[2] == i & 2**24-1
        t |= I[2] == i >> 6
        C[2][t] -= c

        C[3][m] = 0

In [ ]:
from json import dump
dump(O, open('py.64gram5','w'), separators=',:')